# Lab 03: Deploy CertAgent to Amazon Bedrock AgentCore Runtime

## Overview

In this lab, we deploy a persistent AI agent to AgentCore Runtime using the starter toolkit SDK.

**Architecture:**

```
User --> AgentCore Runtime (CertAgent) --> Lambda tools
```

The SDK handles Docker builds (via CodeBuild), ECR, and deployment automatically.

**Estimated time:** 30 minutes

## Install dependencies

In [ ]:
!pip install -q bedrock-agentcore-starter-toolkit strands-agents strands-agents-bedrock boto3

## Load workshop configuration

In [ ]:
import json, pathlib, boto3, os
from boto3.session import Session

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)
REPO_DIR = pathlib.Path(REPO_DIR)

boto_session = Session(region_name=AWS_REGION)
region = AWS_REGION
ssm_client = boto_session.client('ssm')

print(f'Region: {AWS_REGION}')
print(f'Scan Lambda: {LAMBDA_SCAN}')
print('Config loaded')

## Write the CertAgent server code

We create the agent entrypoint that uses the Strands Agents SDK with tools that invoke our Lambda functions.

In [ ]:
%%writefile certagent_server.py
import os
import json
import boto3
from strands import Agent
from strands.models.bedrock import BedrockModel

AWS_REGION = os.environ.get('AWS_REGION', 'us-east-1')
lambda_client = boto3.client('lambda', region_name=AWS_REGION)

LAMBDA_SCAN = os.environ.get('LAMBDA_SCAN', 'certagent-scan-certificates')
LAMBDA_RENEW = os.environ.get('LAMBDA_RENEW', 'certagent-renew-certificate')
LAMBDA_STATUS = os.environ.get('LAMBDA_STATUS', 'certagent-check-status')
LAMBDA_INVENTORY = os.environ.get('LAMBDA_INVENTORY', 'certagent-list-inventory')

def invoke_lambda(fn_name, payload):
    r = lambda_client.invoke(FunctionName=fn_name, InvocationType='RequestResponse',
                             Payload=json.dumps(payload))
    raw = json.loads(r['Payload'].read())
    return raw.get('body', raw)

from strands import tool

@tool
def scan_certificates(threshold_days: int = 30, use_mock: bool = True) -> str:
    """Scan for certificates expiring within the given threshold days.
    Args:
        threshold_days: Number of days to look ahead for expiring certs.
        use_mock: Use mock data instead of real DigiCert API.
    """
    result = invoke_lambda(LAMBDA_SCAN, {'threshold_days': threshold_days, 'use_mock': use_mock})
    return json.dumps(result, indent=2, default=str)

@tool
def renew_certificate(order_id: str, common_name: str, use_mock: bool = True) -> str:
    """Renew a certificate by order ID and domain name.
    Args:
        order_id: The DigiCert order ID to renew.
        common_name: The domain name of the certificate.
        use_mock: Use mock mode for the workshop.
    """
    result = invoke_lambda(LAMBDA_RENEW, {
        'order_id': order_id, 'common_name': common_name,
        'sans': [common_name], 'use_mock': use_mock
    })
    return json.dumps(result, indent=2, default=str)

@tool
def list_inventory(status: str = "all") -> str:
    """List the certificate inventory with optional status filter.
    Args:
        status: Filter by status: all, pending, submitted, issued, completed.
    """
    result = invoke_lambda(LAMBDA_INVENTORY, {'status': status})
    return json.dumps(result, indent=2, default=str)

SYSTEM_PROMPT = """You are CertAgent, an AI operations agent that manages TLS/SSL certificate lifecycle.
Capabilities: SCAN (find expiring certs), RENEW (submit renewal), INVENTORY (view all certs).
Rules:
- Always use mock mode (use_mock=true) in this workshop
- Report priority levels: EXPIRED, CRITICAL (<7d), HIGH (<14d), MEDIUM (<30d), LOW
- Be concise and operational
- After any action, state what was done and what happens next"""

model = BedrockModel(model_id="anthropic.claude-sonnet-4-20250514-v1:0", region_name=AWS_REGION)
agent = Agent(model=model, system_prompt=SYSTEM_PROMPT,
              tools=[scan_certificates, renew_certificate, list_inventory])

def handler(event, context=None):
    input_text = event.get('inputText', event.get('prompt', ''))
    if not input_text:
        return {'output': 'Please provide a message.'}
    response = agent(input_text)
    return {'output': str(response)}

if __name__ == '__main__':
    import sys
    prompt = ' '.join(sys.argv[1:]) if len(sys.argv) > 1 else 'What certs are expiring?'
    print(handler({'inputText': prompt})['output'])

## Write requirements.txt for the agent

In [ ]:
%%writefile certagent_requirements.txt
strands-agents>=0.1.0
strands-agents-bedrock>=0.1.0
boto3>=1.35.0

## Configure AgentCore Runtime

Use the starter toolkit to configure the deployment. This generates a Dockerfile and sets up ECR + IAM roles automatically.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agent_name = "certagent"

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="certagent_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="certagent_requirements.txt",
    region=region,
    protocol="HTTP",
    agent_name=agent_name,
)
print("Configuration completed")

## Launch to AgentCore Runtime

This builds the container via CodeBuild (ARM64), pushes to ECR, and deploys to AgentCore Runtime. Takes ~3-5 minutes.

In [ ]:
print("Launching CertAgent to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

# Store ARN in SSM for later use
ssm_client.put_parameter(
    Name=f'/workshop/{config.get("WORKSHOP_PREFIX", "certagent")}/agent-arn',
    Value=launch_result.agent_arn,
    Type='String',
    Overwrite=True
)
print("Agent ARN stored in SSM")

## Invoke the deployed agent

Now invoke the agent running in AgentCore Runtime using the SDK.

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

# Reload if needed
rt = Runtime()
print("Invoking CertAgent...")
response = rt.invoke("What certificates are expiring soon? Use mock mode and give a prioritized summary.")
print(response)

In [ ]:
print("Invoking: renew api.example.com...")
response = rt.invoke("Renew the certificate for api.example.com using mock mode")
print(response)

In [ ]:
print("Invoking: multi-step scan + renew...")
response = rt.invoke("Scan for certs expiring in 7 days with mock data, then renew any CRITICAL ones")
print(response)

In [ ]:
print("Invoking: show inventory...")
response = rt.invoke("Show the full certificate inventory grouped by renewal status")
print(response)

## View agent status and logs

In [ ]:
# Check agent status
status = rt.status()
print(f"Agent status: {status}")

## Lab 03 Complete

You have:
1. Written a Strands-based agent with Lambda-backed tools
2. Configured the AgentCore Runtime deployment (Docker, ECR, IAM)
3. Deployed the agent via CodeBuild to AgentCore Runtime
4. Invoked the persistent agent with scan, renew, and multi-step operations

The agent is now running as a managed service in AgentCore Runtime.

**Next:** `04_proactive_monitoring.ipynb`